# Trabajo Práctico N° 2 - Teledetección
## Clasificación de Imágenes Satelitales para la Detección y Cuantificación de Áreas Inundadas (Bahía Blanca, 2025)

**Docentes:** Federico Bayle y Carolina S. Ramos  
**Estudiante:** Augusto Rey  
**Materia:** Sistemas de Información Geográfica (Maestría en Explotación de Datos y Descubrimiento del Conocimiento, UBA)  

---

### Objetivo
El objetivo de este notebook es guiar a través de los pasos técnicos de procesamiento de imágenes satelitales Sentinel-2 para:
1. Calcular índices espectrales de agua (NDWI y MNDWI).
2. Implementar algoritmos de clasificación de agua (Random Forest semi-supervisado y K-Means no supervisado).
3. Integrar datos externos de elevación (Copernicus DEM) y población (WorldPop).
4. Filtrar y validar los resultados usando capas oficiales de hidrología del IGN.
5. Cuantificar las hectáreas inundadas y la población expuesta en la ciudad de Bahía Blanca en el año 2025.

### 0. Importación de Librerías y Configuración de Entorno

In [ ]:
import os
import sys
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path

# Añadir directorio de scripts para importar procesamiento.py
sys.path.append(str(Path("/home/augusto/Desktop/TP2/scripts")))

# Importar las funciones de nuestro script de procesamiento
import procesamiento as pr

BASE_DIR = Path("/home/augusto/Desktop/TP2")
DATA_DIR = BASE_DIR / "data-Sentinel-2"
AOI_FILE = DATA_DIR / "aoi.geojson"
IMG_FEB = DATA_DIR / "AOI_20250219_20m_clip.tif"
IMG_MAR = DATA_DIR / "AOI_20250311_20m_clip.tif"

print("Configuración cargada y librerías importadas con éxito.")

### 1. Descarga y Preparación Automática de Datos Externos
Descargamos de forma automática el **DEM de Copernicus GLO-30** (desde AWS), el raster de población de **WorldPop 1km**, y las capas de hidrología del **IGN** a través del WFS oficial.

In [ ]:
print("Descargando DEM y Población...")
pr.download_dem_and_pop()

print("Descargando Capas de Hidrología del IGN (WFS)...")
pr.download_ign_hydrology()

print("Preprocesando, uniendo y alineaando rasters...")
pr.preprocess_rasters()

### 2. Inspección Visual de Imágenes Sentinel-2 y Cálculo de Índices Espectrales (NDWI y MNDWI)
Calculamos y visualizamos el índice espectral MNDWI para ambas fechas.

In [ ]:
# Leer imágenes
bands_feb = pr.read_sentinel_image(IMG_FEB)
bands_mar = pr.read_sentinel_image(IMG_MAR)

# Calcular índices
ndwi_feb, mndwi_feb = pr.calculate_indices(bands_feb)
ndwi_mar, mndwi_mar = pr.calculate_indices(bands_mar)

# Graficar comparación de MNDWI
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(mndwi_feb, cmap="RdYlBu", vmin=-0.6, vmax=0.6)
axes[0].set_title("MNDWI - 19 de Febrero 2025 (Pre-evento)")
axes[0].axis("off")

im1 = axes[1].imshow(mndwi_mar, cmap="RdYlBu", vmin=-0.6, vmax=0.6)
axes[1].set_title("MNDWI - 11 de Marzo 2025 (Post-evento)")
axes[1].axis("off")

fig.colorbar(im1, ax=axes.ravel().tolist(), label="MNDWI", shrink=0.8)
plt.show()

### 3. Clasificación de Agua mediante Machine Learning (Random Forest y K-Means)
Entrenamos de forma automática un clasificador **Random Forest** semi-supervisado y lo comparamos con un agrupamiento no supervisado **K-Means** (K=5).

In [ ]:
print("Ejecutando clasificación semi-supervisada Random Forest para Febrero...")
water_feb_rf, _, _ = pr.classify_water_random_forest(IMG_FEB)

print("Ejecutando clasificación semi-supervisada Random Forest para Marzo...")
water_mar_rf, _, _ = pr.classify_water_random_forest(IMG_MAR)

print("Ejecutando agrupamiento no supervisado K-Means para Marzo...")
water_mar_km = pr.classify_water_kmeans(IMG_MAR)

# Graficar comparación de clasificadores
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(water_mar_rf, cmap="Blues")
axes[0].set_title("Clasificación de Agua: Random Forest (Marzo 2025)")
axes[0].axis("off")

axes[1].imshow(water_mar_km, cmap="Blues")
axes[1].set_title("Clasificación de Agua: K-Means (Marzo 2025)")
axes[1].axis("off")
plt.show()

### 4. Filtrado Topográfico e Integración Hidrología del IGN
Filtramos falsos positivos eliminando celdas ubicadas en pendientes empinadas o altitudes incoherentes utilizando el DEM, y descontando los cuerpos de agua permanentes del IGN.

In [ ]:
# Detectar cambio (inundación cruda)
raw_inundation = (water_mar_rf > 0) & ~(water_feb_rf > 0)

# Aplicar filtro topográfico (DEM)
dem_path = DATA_DIR / "dem_clip.tif"
filtered_inundation, dem, slope = pr.apply_topographic_filter(raw_inundation, dem_path)

# Cargar capas de hidrología IGN y crear máscara de agua permanente
ign_path = DATA_DIR / "ign_hydrology.geojson"
ign_water_mask = np.zeros_like(filtered_inundation, dtype=bool)
if ign_path.exists():
    ign_gdf = gpd.read_file(ign_path)
    ign_areas = ign_gdf[ign_gdf['layer'] == 'ign:areas_de_aguas_continentales_perenne']
    if len(ign_areas) > 0:
        with rasterio.open(IMG_FEB) as src:
            ign_areas_proj = ign_areas.to_crs(src.crs)
            ign_water_mask_raw, _ = rasterio.mask.mask(src, ign_areas_proj.geometry, invert=False, filled=True)
            ign_water_mask = (ign_water_mask_raw[0] > 0)

# Inundación limpia final
clean_inundation = filtered_inundation & (~ign_water_mask)

print("Filtrado completado con éxito.")

### 5. Resultados Finales e Impacto de Población
Calculamos las hectáreas finales inundadas y cruzamos el resultado con WorldPop para calcular las personas expuestas.

In [ ]:
# Cada pixel de 20x20m = 0.04 ha
pixel_area_ha = 0.04
hectares_affected = np.sum(clean_inundation) * pixel_area_ha

# Cargar población y calcular expuestos
pop_path = DATA_DIR / "worldpop_clip.tif"
with rasterio.open(pop_path) as src:
    pop_density = src.read(1)
    pop_density = np.where(pop_density < 0, 0, pop_density)

# Ajustar por densidad espacial de 20m vs 1km (proporción de área 400m2 / 1000000m2 = 0.0004)
poblacion_afectada = np.sum(pop_density[clean_inundation]) * 0.0004

print(f"Hectáreas totales inundadas detectadas: {hectares_affected:.2f} ha")
print(f"Habitantes expuestos estimados: {int(poblacion_afectada)} personas")

### 6. Ejecución del Dashboard Interactivo
Para abrir el visor interactivo de Streamlit, ejecute la siguiente línea de comando en su terminal:
```bash
streamlit run scripts/dashboard.py
```